# Análisis Exploratorio (EDA) — Superstore: Pandas vs PySpark

**Autor:** Eduardo Osorio Venegas
**Dataset:** `SuperStore_Tablon.xlsx` (9.800 pedidos, 18 columnas, 2015–2018)

Este notebook hace el **mismo análisis exploratorio dos veces emplando python**: una con **pandas** y otra con **PySpark**, para que se vea lado a lado cómo cambia la sintaxis — y, más importante, **por qué** cambia.

**La diferencia de fondo entre ambos:**

| | pandas | PySpark |
|---|---|---|
| Motor de ejecución | En memoria, en un solo proceso Python | Distribuido, en un cluster (varios executors) |
| Evaluación | **Eager**: cada línea se ejecuta al momento | **Lazy**: arma un plan y ejecuta recién con una acción (`.show()`, `.count()`, `.collect()`) |
| Estructura de datos | `DataFrame` con **índice** | `DataFrame` **sin índice** (son filas distribuidas) |
| Tamaño de datos ideal | Lo que entre en la memoria de una máquina | Escala a terabytes, repartido entre nodos |
| Cuándo conviene en Fabric | Archivos chicos/medianos, prototipado rápido, Excel/CSV que no requieren distribución | Tablas grandes del Lakehouse/Warehouse, transformaciones que van a producción |

Superstore (9.800 filas) es un archivo pequeño de datos — pandas lo procesa sin problema. Lo usamos igual con PySpark acá **a modo didáctico**, para comparar sintaxis; en un caso real de este tamaño no haría falta Spark.


## 0. Setup — leer el mismo archivo con las dos herramientas


In [2]:
import pandas as pd
from pyspark.sql import functions as F

ruta_archivo = "/lakehouse/default/Files/EDUARDO OSORIO/XLSX/SuperStore_Tablon.xlsx"

# --- pandas: DataFrame en memoria, con índice ---
pdf = pd.read_excel(ruta_archivo)
# --- pandas: Ajustar tipo de datos de columnas ---
pdf["Order_Date"] = pd.to_datetime(pdf["Order_Date"])
pdf["Ship_Date"] = pd.to_datetime(pdf["Ship_Date"])

# --- PySpark: mismo contenido, como DataFrame distribuido ---
df = spark.createDataFrame(pdf)

print(f"pandas  -> {type(pdf)}")
print(f"PySpark -> {type(df)}")


StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 4, Finished, Available, Finished, False)

pandas  -> <class 'pandas.core.frame.DataFrame'>
PySpark -> <class 'pyspark.sql.dataframe.DataFrame'>


## 1. Dimensiones del dataset

En pandas, `.shape` da la respuesta al instante porque el DataFrame ya está completo en memoria. En PySpark, `.count()` es una **acción**: dispara la ejecución del plan sobre el cluster — por eso es la operación más "cara" de esta sección, aunque acá no se note con 9.800 filas.


In [5]:
# pandas — eager, inmediato
print(f"Shape: {pdf.shape}" )
n_filas, n_cols = pdf.shape
print(f"pandas: {n_filas} filas x {n_cols} columnas")

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 7, Finished, Available, Finished, False)

Shape: (9800, 18)
pandas: 9800 filas x 18 columnas


In [4]:
# PySpark — count() es una acción (dispara el plan lazy)
n_filas = df.count()
n_cols = len(df.columns)
print(f"PySpark: {n_filas} filas x {n_cols} columnas")

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 6, Finished, Available, Finished, False)

PySpark: 9800 filas x 18 columnas


## 2. Esquema y tipos de datos

`dtypes` en pandas es una Series de NumPy/pandas dtypes. `printSchema()` en Spark muestra el árbol de tipos Spark SQL (`StringType`, `DoubleType`, etc.) — son sistemas de tipos distintos, así que un mismo `float64` de pandas puede mapear a `DoubleType` en Spark.


In [6]:
# pandas
print(pdf.dtypes)


StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 9, Finished, Available, Finished, False)

Row_ID                    int64
Order_ID                 object
Order_Date       datetime64[ns]
Ship_Date        datetime64[ns]
Ship_Mode                object
Customer_ID              object
Customer_Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal_Code             float64
Region                   object
Product_ID               object
Category                 object
Sub_Category             object
Product_Name             object
Sales                   float64
dtype: object


In [7]:
pdf.info()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 10, Finished, Available, Finished, False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row_ID         9800 non-null   int64         
 1   Order_ID       9800 non-null   object        
 2   Order_Date     9800 non-null   datetime64[ns]
 3   Ship_Date      9800 non-null   datetime64[ns]
 4   Ship_Mode      9800 non-null   object        
 5   Customer_ID    9800 non-null   object        
 6   Customer_Name  9800 non-null   object        
 7   Segment        9800 non-null   object        
 8   Country        9800 non-null   object        
 9   City           9800 non-null   object        
 10  State          9800 non-null   object        
 11  Postal_Code    9789 non-null   float64       
 12  Region         9800 non-null   object        
 13  Product_ID     9800 non-null   object        
 14  Category       9800 non-null   object        
 15  Sub_Category   9800 n

In [8]:
# PySpark
df.printSchema()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 11, Finished, Available, Finished, False)

root
 |-- Row_ID: long (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Ship_Date: timestamp (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: double (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)



## 3. Primeras filas

`head()` devuelve un objeto pandas que se renderiza como tabla directamente. En Spark, `show()` imprime texto plano por consola; `display(df)` función propia de una grilla interactiva más parecida a lo que se ve en pandas.


In [9]:
# pandas
pdf.head(5)

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 13, Finished, Available, Finished, False)

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62
3,4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O Donnel,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,9575775.00
4,5,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O Donnel,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold N Roll Cart System,22368.00


In [10]:
# PySpark
df.show(5, truncate=False)

# En Fabric, la alternativa "linda" es:
display(df.limit(5))

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 15, Finished, Available, Finished, False)

+------+--------------+-------------------+-------------------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+---------+
|Row_ID|Order_ID      |Order_Date         |Ship_Date          |Ship_Mode     |Customer_ID|Customer_Name  |Segment  |Country      |City           |State     |Postal_Code|Region|Product_ID     |Category       |Sub_Category|Product_Name                                               |Sales    |
+------+--------------+-------------------+-------------------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+---------+
|1     |CA-2017-152156|2017-11-08 00:00:00|2017-11-11 00:00:00|Second Class  |CG-12520   |Claire Gute    |Consumer |United S

In [11]:
display(df.limit(5))

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f718772-921e-49d7-9356-66415375b3ff)

## 4. Valores nulos por columna

En pandas es una sola cadena de métodos vectorizados. En Spark no existe un `.isnull().sum()` directo sobre todas las columnas: hay que construir la expresión por columna con una comprensión de listas.


In [12]:
# pandas
pdf.isnull().sum()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 17, Finished, Available, Finished, False)

Row_ID            0
Order_ID          0
Order_Date        0
Ship_Date         0
Ship_Mode         0
Customer_ID       0
Customer_Name     0
Segment           0
Country           0
City              0
State             0
Postal_Code      11
Region            0
Product_ID        0
Category          0
Sub_Category      0
Product_Name      0
Sales             0
dtype: int64

In [13]:
# PySpark
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show(vertical=True)

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 18, Finished, Available, Finished, False)

-RECORD 0------------
 Row_ID        | 0   
 Order_ID      | 0   
 Order_Date    | 0   
 Ship_Date     | 0   
 Ship_Mode     | 0   
 Customer_ID   | 0   
 Customer_Name | 0   
 Segment       | 0   
 Country       | 0   
 City          | 0   
 State         | 0   
 Postal_Code   | 11  
 Region        | 0   
 Product_ID    | 0   
 Category      | 0   
 Sub_Category  | 0   
 Product_Name  | 0   
 Sales         | 0   



En este dataset, solo `Postal_Code` tiene nulos (11 de 9.800 filas) — el resto de las columnas están completas.

## 5. Cardinalidad de las columnas categóricas

`nunique()` es directo en pandas. En Spark, `countDistinct()` es exacto pero cuesta un shuffle completo de los datos; para tablas realmente grandes se suele usar `approx_count_distinct()`, que es aproximado pero mucho más barato computacionalmente.


In [14]:
# pandas
for col in ["Category", "Sub_Category", "Region", "Segment"]:
    print(f"{col}: {pdf[col].nunique()} valores únicos")

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 19, Finished, Available, Finished, False)

Category: 3 valores únicos
Sub_Category: 17 valores únicos
Region: 4 valores únicos
Segment: 3 valores únicos


In [18]:
pdf["Category"].value_counts()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 25, Finished, Available, Finished, False)

Category
Office Supplies    5909
Furniture          2078
Technology         1813
Name: count, dtype: int64

In [19]:
# PySpark — exacto (countDistinct) vs aproximado (approx_count_distinct)
df.select([
    F.countDistinct(c).alias(f"{c}_exact") for c in ["Category", "Sub_Category", "Region", "Segment"]
]).show()

df.select([
    F.approx_count_distinct(c).alias(f"{c}_approx") for c in ["Category", "Sub_Category", "Region", "Segment"]
]).show()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 26, Finished, Available, Finished, False)

+--------------+------------------+------------+-------------+
|Category_exact|Sub_Category_exact|Region_exact|Segment_exact|
+--------------+------------------+------------+-------------+
|             3|                17|           4|            3|
+--------------+------------------+------------+-------------+

+---------------+-------------------+-------------+--------------+
|Category_approx|Sub_Category_approx|Region_approx|Segment_approx|
+---------------+-------------------+-------------+--------------+
|              3|                 16|            4|             3|
+---------------+-------------------+-------------+--------------+



## 6. Frecuencia de categorías

`value_counts()` ordena de mayor a menor por defecto. En Spark hace falta encadenar `groupBy().count()` y ordenar explícitamente con `orderBy()`.


In [20]:
# pandas
pdf["Category"].value_counts()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 27, Finished, Available, Finished, False)

Category
Office Supplies    5909
Furniture          2078
Technology         1813
Name: count, dtype: int64

In [21]:
# PySpark
df.groupBy("Category").count().orderBy(F.desc("count")).show()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 28, Finished, Available, Finished, False)

+---------------+-----+
|       Category|count|
+---------------+-----+
|Office Supplies| 5909|
|      Furniture| 2078|
|     Technology| 1813|
+---------------+-----+



Resultado en ambos casos: **Office Supplies** domina en cantidad de pedidos (5.909), seguido de Furniture (2.078) y Technology (1.813).


## 7. Estadísticas descriptivas de `Sales`

`describe()` existe en las dos herramientas con nombre casi idéntico, pero el resultado de Spark viene como DataFrame de strings (hay que castear si se quiere operar con los números) y no incluye percentiles por defecto — hace falta `summary()` para eso.


In [22]:
# pandas — incluye percentiles (25/50/75) por defecto
pdf["Sales"].describe()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 29, Finished, Available, Finished, False)

count    9.800000e+03
mean     1.014267e+05
std      5.215577e+05
min      4.440000e-01
25%      3.989500e+01
50%      3.563750e+02
75%      2.534400e+04
max      2.396266e+07
Name: Sales, dtype: float64

In [23]:
# PySpark — describe() no trae percentiles; summary() sí
df.select("Sales").describe().show()
df.select("Sales").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 30, Finished, Available, Finished, False)

+-------+------------------+
|summary|             Sales|
+-------+------------------+
|  count|              9800|
|   mean|101426.65048326526|
| stddev| 521557.7453120213|
|    min|             0.444|
|    max|       2.3962656E7|
+-------+------------------+

+-------+------------------+
|summary|             Sales|
+-------+------------------+
|  count|              9800|
|   mean|101426.65048326526|
| stddev| 521557.7453120213|
|    min|             0.444|
|    25%|             39.88|
|    50%|            355.36|
|    75%|           25344.0|
|    max|       2.3962656E7|
+-------+------------------+



## 8. Ventas totales por categoría

Misma lógica de agregación en ambos — acá la sintaxis es la que más se parece entre las dos herramientas.


In [24]:
# pandas
pdf.groupby("Category")["Sales"].sum().sort_values(ascending=False)

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 31, Finished, Available, Finished, False)

Category
Furniture          5.115324e+08
Technology         2.757589e+08
Office Supplies    2.066898e+08
Name: Sales, dtype: float64

In [25]:
# PySpark
(
    df.groupBy("Category")
    .agg(F.sum("Sales").alias("Sales_total"))
    .orderBy(F.desc("Sales_total"))
    .show()
)

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 32, Finished, Available, Finished, False)

+---------------+--------------------+
|       Category|         Sales_total|
+---------------+--------------------+
|      Furniture|      5.1153240282E8|
|     Technology|       2.757589222E8|
|Office Supplies|2.0668984971600017E8|
+---------------+--------------------+



Ojo con el orden: por **cantidad de pedidos** Office Supplies gana, pero por **monto total vendido** el líder es **Furniture** — la categoría con menos pedidos pero de mayor ticket promedio.


## 9. Top N por una columna

`nlargest()` es un atajo directo en pandas. En Spark se arma con `orderBy()` descendente + `limit()` — no hay un método "top N" dedicado.


In [26]:
# pandas
pdf.nlargest(5, "Sales")[["Row_ID", "Product_Name", "Sales"]]

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 33, Finished, Available, Finished, False)

,Row_ID,Product_Name,Sales
399,400,"Riverside Palais Royal Lawyers Bookcase, Royal...",23962656.0
5055,5056,OSullivan Living Dimensions 5-Shelf Bookcases,13523976.0
8781,8782,"Atlantic Metals Mobile 5-Shelf Bookcases, Cust...",12279984.0
2623,2624,Canon imageCLASS 2200 Advanced Copier,11199968.0
3,4,Bretford CR4500 Series Slim Rectangular Table,9575775.0


In [27]:
# PySpark
(
    df.orderBy(F.desc("Sales"))
    .select("Row_ID", "Product_Name", "Sales")
    .limit(5)
    .show(truncate=False)
)

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 34, Finished, Available, Finished, False)

+------+-------------------------------------------------------------+-----------+
|Row_ID|Product_Name                                                 |Sales      |
+------+-------------------------------------------------------------+-----------+
|400   |Riverside Palais Royal Lawyers Bookcase, Royale Cherry Finish|2.3962656E7|
|5056  |OSullivan Living Dimensions 5-Shelf Bookcases                |1.3523976E7|
|8782  |Atlantic Metals Mobile 5-Shelf Bookcases, Custom Colors      |1.2279984E7|
|2624  |Canon imageCLASS 2200 Advanced Copier                        |1.1199968E7|
|4     |Bretford CR4500 Series Slim Rectangular Table                |9575775.0  |
+------+-------------------------------------------------------------+-----------+



## 10. Filtrado condicional

El *boolean indexing* de pandas (`df[condición]`) es el equivalente directo al `.filter()` / `.where()` de Spark (son alias, hacen lo mismo).


In [28]:
# pandas — boolean indexing
pdf[(pdf["Category"] == "Technology") & (pdf["Sales"] > 1000)].shape[0]

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 35, Finished, Available, Finished, False)

805

In [29]:
# PySpark — .filter() (o .where(), son equivalentes)
df.filter((F.col("Category") == "Technology") & (F.col("Sales") > 1000)).count()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 36, Finished, Available, Finished, False)

805

## 11. Crear una columna derivada

`assign()` (o asignación directa) es vectorizado y eager en pandas. `withColumn()` en Spark solo agrega un paso al plan lazy — no se materializa hasta la próxima acción.


In [30]:
# pandas
pdf["Dias_envio"] = (pdf["Ship_Date"] - pdf["Order_Date"]).dt.days
pdf[["Order_Date", "Ship_Date", "Dias_envio"]].head()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 37, Finished, Available, Finished, False)

,Order_Date,Ship_Date,Dias_envio
0,2017-11-08,2017-11-11,3
1,2017-11-08,2017-11-11,3
2,2017-06-12,2017-06-16,4
3,2016-10-11,2016-10-18,7
4,2016-10-11,2016-10-18,7


In [31]:
# PySpark
df = df.withColumn("Dias_envio", F.datediff(F.col("Ship_Date"), F.col("Order_Date")))
df.select("Order_Date", "Ship_Date", "Dias_envio").show(5)

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 38, Finished, Available, Finished, False)

+-------------------+-------------------+----------+
|         Order_Date|          Ship_Date|Dias_envio|
+-------------------+-------------------+----------+
|2017-11-08 00:00:00|2017-11-11 00:00:00|         3|
|2017-11-08 00:00:00|2017-11-11 00:00:00|         3|
|2017-06-12 00:00:00|2017-06-16 00:00:00|         4|
|2016-10-11 00:00:00|2016-10-18 00:00:00|         7|
|2016-10-11 00:00:00|2016-10-18 00:00:00|         7|
+-------------------+-------------------+----------+
only showing top 5 rows



## 12. Correlación entre columnas numéricas

Acá es donde más se nota la diferencia de filosofía: pandas resuelve una matriz de correlación completa con **una sola línea**. Spark no tiene ese atajo — hay que armar un vector de features con `VectorAssembler` y usar `Correlation.corr()` de `pyspark.ml`, mucho más manual.


In [32]:
# pandas — matriz de correlación completa en una línea
pdf[["Sales", "Postal_Code"]].corr()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 39, Finished, Available, Finished, False)

,Sales,Postal_Code
Sales,1.000000,0.031332
Postal_Code,0.031332,1.000000


In [33]:
# PySpark — requiere pyspark.ml para lo mismo
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

ensamblador = VectorAssembler(inputCols=["Sales", "Postal_Code"], outputCol="features")
df_vec = ensamblador.transform(df.na.drop(subset=["Sales", "Postal_Code"]))

matriz_corr = Correlation.corr(df_vec, "features").head()[0]
print(matriz_corr)

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 40, Finished, Available, Finished, False)

DenseMatrix([[1.        , 0.03133153],
             [0.03133153, 1.        ]])


En este dataset la correlación es prácticamente nula (~0.03) — `Postal_Code` es un código postal, no una magnitud continua, así que no se espera relación con `Sales`. El ejercicio es más sobre la sintaxis que sobre el hallazgo en sí.


## 13. Rango de fechas

Mismo patrón que en la sección 7: agregaciones simples (`min`/`max`) se escriben parecido en ambas herramientas.


In [34]:
# pandas
print(pdf["Order_Date"].min(), "->", pdf["Order_Date"].max())

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 41, Finished, Available, Finished, False)

2015-01-03 00:00:00 -> 2018-12-30 00:00:00


In [35]:
# PySpark
df.select(F.min("Order_Date"), F.max("Order_Date")).show()

StatementMeta(, 283268c1-8220-4f2a-a5f7-4e13a72a717e, 42, Finished, Available, Finished, False)

+-------------------+-------------------+
|    min(Order_Date)|    max(Order_Date)|
+-------------------+-------------------+
|2015-01-03 00:00:00|2018-12-30 00:00:00|
+-------------------+-------------------+



## 14. Resumen: ¿pandas o PySpark para EDA?

| Tarea | Ganador para EDA rápido | Por qué |
|---|---|---|
| Dataset chico (cabe en memoria) | **pandas** | Sintaxis más corta, eager, sin overhead de un cluster |
| Dataset grande (Lakehouse/Warehouse) | **PySpark** | Distribuye el cómputo, no depende de la RAM de un solo nodo |
| Correlaciones, pivots, estadística rápida | **pandas** | Muchas más funciones "de una línea" ya resueltas |
| Transformaciones que van a producción sobre datos que van a crecer | **PySpark** | Escala sin reescribir código cuando el volumen aumenta |
| Prototipado / exploración interactiva | **pandas** | Eager: cada celda muestra el resultado al instante |

**Regla práctica en Fabric:** para explorar un archivo que llega como Excel/CSV chico, pandas es más rápido de escribir. Para explorar una tabla del Lakehouse que ya tiene millones de filas, PySpark es la herramienta correcta — y de hecho `pdf = df.toPandas()` (traer una muestra a pandas) es un patrón común cuando se necesita esa sintaxis más rica solo para explorar un subconjunto pequeño.
